# FinChart-R2 — Phase 2C Unsloth Multimodal DPO-386 Pilot

This Colab notebook continues from the SFT-408 adapter `Kxck/Finance_500_v1` and performs **offline multimodal Direct Preference Optimization (DPO)** on ChartQA train-only preference pairs.

Research boundary:

```text
ChartQA train[500:2500] -> SFT error mining -> teacher capture
-> deterministic gates -> 386 provisional pairs -> DPO pilot
```

The frozen ChartQA `val[0:500]` subset is not loaded here. The current 386 pairs remain provisional until manual audit. Running them with `ALLOW_PROVISIONAL_PAIRS=True` creates a diagnostic adapter, not a final reportable model.

Unsloth loads and optimizes the 4-bit Qwen3-VL policy. TRL still owns the native vision preference collator; an explicit preflight blocks training unless real image tensors are present. Official references: [TRL DPOTrainer](https://huggingface.co/docs/trl/main/dpo_trainer), [Qwen3-VL](https://huggingface.co/docs/transformers/model_doc/qwen3_vl), and the [Unsloth Qwen3-VL 4-bit checkpoint](https://huggingface.co/unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit).

## 1. Install a VLM-DPO-compatible stack

Run this cell once in a fresh Colab GPU runtime, then restart the runtime before continuing. Do not import `torch`, `transformers`, `trl`, or `peft` before Unsloth in the restarted runtime.

In [ ]:
%pip install -q -U --no-cache-dir unsloth unsloth_zoo trl peft transformers datasets accelerate bitsandbytes huggingface_hub "pillow>=11.3.0,<13.0.0" qwen-vl-utils
print('Installation complete. Restart the runtime once, then continue from Section 2.')

## 2. Runtime and version gate

In [ ]:
# Unsloth must be imported before torch, transformers, trl, and peft.
import unsloth

import json
import os
import platform
import random
import warnings
from collections import Counter
from pathlib import Path

import datasets
import peft
import torch
import transformers
import trl
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing.')
try:
    from trl.trainer.dpo_trainer import DataCollatorForVisionPreference
except ImportError as error:
    raise RuntimeError(
        f'TRL {trl.__version__} has no native vision preference collator. '
        'Restart from a fresh runtime and rerun the install cell.'
    ) from error

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
gpu = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.is_bf16_supported()
print({
    'python': platform.python_version(),
    'unsloth': getattr(unsloth, '__version__', 'unknown'),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'trl': trl.__version__,
    'peft': peft.__version__,
    'datasets': datasets.__version__,
    'gpu': gpu.name,
    'gpu_memory_GB': round(gpu.total_memory / 2**30, 1),
    'bf16': BF16,
})

## 3. Mount Drive and configure the pilot

Upload `phase2c_teacher_v1_dpo_candidates_provisional.jsonl` when prompted if it is not already in the Drive data directory. For a final experiment, replace it with the manually audited JSONL and set `ALLOW_PROVISIONAL_PAIRS=False`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

BASE_MODEL = 'Qwen/Qwen3-VL-4B-Instruct'
UNSLOTH_BASE_MODEL = 'unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit'
MODEL_BACKEND = 'UNSLOTH'  # Unsloth loader + native TRL VLM DPO collator
SFT_ADAPTER_ID = 'Kxck/Finance_500_v1'
DATASET_NAME = 'HuggingFaceM4/ChartQA'

RUN_ROOT = Path('/content/drive/MyDrive/FinChart-R2/phase2c/dpo_386_provisional_pilot')
DATA_DIR = RUN_ROOT / 'data'
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
ADAPTER_DIR = RUN_ROOT / 'adapter_sft_dpo_386'
PAIR_JSONL = DATA_DIR / 'phase2c_teacher_v1_dpo_candidates_provisional.jsonl'

EXPECTED_PAIRS = 386
PAIR_EVAL_FRACTION = 0.10
APPROVED_AUDIT_STATUSES = {'APPROVED', 'VALIDATED', 'AUDITED_APPROVED'}
ALLOW_PROVISIONAL_PAIRS = True  # diagnostic only; False for a reportable audited run
ALLOW_SCHEMA_MISMATCH = True  # current rejected responses are often truncated or answer-only
RUN_TRAINING = True
RESUME_FROM_LAST_CHECKPOINT = True
PUSH_TO_HUB = False
HUB_MODEL_ID = 'Kxck/Finance_500_v1_DPO_386'

for directory in (DATA_DIR, CHECKPOINT_DIR, ADAPTER_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print('Pair file:', PAIR_JSONL)
print('Output root:', RUN_ROOT)

## 4. Load and gate the preference pairs

This gate verifies train-only provenance, uniqueness, non-empty preferences, manual-audit state, and response-field symmetry. Schema asymmetry is reported because it can create a format preference unrelated to correctness.

In [ ]:
import hashlib
import re
import shutil

if not PAIR_JSONL.exists():
    from google.colab import files
    print('Upload the provisional or manually audited DPO JSONL.')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('No DPO pair file was uploaded.')
    uploaded_name = next(iter(uploaded))
    source_path = Path('/content') / uploaded_name
    source_path.write_bytes(uploaded[uploaded_name])
    shutil.copy2(source_path, PAIR_JSONL)

records = [
    json.loads(line)
    for line in PAIR_JSONL.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
required = {
    'dataset_index', 'image_split', 'image_index', 'prompt',
    'chosen', 'rejected', 'ground_truth', 'manual_audit_status'
}
for line_number, row in enumerate(records, 1):
    missing = required - set(row)
    if missing:
        raise ValueError(f'Pair line {line_number} is missing {sorted(missing)}')
    if row['image_split'] != 'train':
        raise RuntimeError(f'Validation leakage at pair line {line_number}: {row["image_split"]}')
    if not all(str(row[key]).strip() for key in ('prompt', 'chosen', 'rejected')):
        raise ValueError(f'Empty prompt/chosen/rejected at pair line {line_number}')
    if str(row['chosen']).strip() == str(row['rejected']).strip():
        raise ValueError(f'Identical chosen and rejected at pair line {line_number}')

if EXPECTED_PAIRS is not None and len(records) != EXPECTED_PAIRS:
    raise ValueError(f'Expected {EXPECTED_PAIRS} pairs, found {len(records)}')
indices = [int(row['dataset_index']) for row in records]
image_refs = {(row['image_split'], int(row['image_index'])) for row in records}
if len(set(indices)) != len(records) or len(image_refs) != len(records):
    raise ValueError('Duplicate dataset indices or image references detected.')

audit_statuses = Counter(str(row['manual_audit_status']) for row in records)
pending_count = sum(
    count for status, count in audit_statuses.items()
    if status not in APPROVED_AUDIT_STATUSES
)
if pending_count and not ALLOW_PROVISIONAL_PAIRS:
    raise RuntimeError(
        f'{pending_count} pairs are not APPROVED. Complete manual audit or explicitly enable diagnostic mode.'
    )

required_fields = ('Relevant values', 'Operation', 'Calculation', 'Answer')
def field_signature(text):
    return tuple(
        field for field in required_fields
        if re.search(rf'^\s*{re.escape(field)}\s*:', str(text), re.I | re.M)
    )
chosen_full = sum(field_signature(row['chosen']) == required_fields for row in records)
rejected_full = sum(field_signature(row['rejected']) == required_fields for row in records)
schema_matched = sum(
    field_signature(row['chosen']) == field_signature(row['rejected']) for row in records
)
pair_sha256 = hashlib.sha256(PAIR_JSONL.read_bytes()).hexdigest()
print({
    'records': len(records),
    'unique_dataset_indices': len(set(indices)),
    'unique_images': len(image_refs),
    'split': sorted({row['image_split'] for row in records}),
    'manual_audit_status': dict(audit_statuses),
    'chosen_full_4_fields': chosen_full,
    'rejected_full_4_fields': rejected_full,
    'same_field_signature': schema_matched,
    'sha256': pair_sha256,
})
if pending_count:
    warnings.warn('PROVISIONAL DIAGNOSTIC: manual audit is incomplete; do not report this adapter as final.')
if schema_matched < len(records) and not ALLOW_SCHEMA_MISMATCH:
    raise RuntimeError(
        f'Only {schema_matched}/{len(records)} pairs share the same response-field signature.'
    )
if schema_matched < len(records):
    warnings.warn(
        f'FORMAT-BIAS RISK: only {schema_matched}/{len(records)} pairs have the same field signature.'
    )

## 5. Rehydrate ChartQA images and build the TRL vision preference dataset

The JSONL stores stable ChartQA split/index references instead of duplicating images. Native TRL expects a single `image` column plus conversational `prompt`, `chosen`, and `rejected` fields. The collator injects the image into the prompt at batch time.

In [ ]:
from datasets import Dataset, load_dataset

chartqa_train = load_dataset(DATASET_NAME, split='train')
examples = []
for row in records:
    image_index = int(row['image_index'])
    source = chartqa_train[image_index]
    source_question = str(source.get('query', source.get('question', ''))).strip()
    if source_question != str(row['prompt']).strip():
        raise ValueError(f'ChartQA question mismatch at dataset_index={row["dataset_index"]}')
    examples.append({
        'image': source['image'].convert('RGB'),
        'prompt': [{'role': 'user', 'content': str(row['prompt'])}],
        'chosen': [{'role': 'assistant', 'content': str(row['chosen'])}],
        'rejected': [{'role': 'assistant', 'content': str(row['rejected'])}],
    })

preference_dataset = Dataset.from_list(examples)
split_dataset = preference_dataset.train_test_split(
    test_size=PAIR_EVAL_FRACTION, seed=SEED, shuffle=True
)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']
print(preference_dataset)
print({'train_pairs': len(train_dataset), 'eval_pairs': len(eval_dataset)})
print('Prompt example:', records[0]['prompt'])
print('Chosen example:\n' + records[0]['chosen'])
print('Rejected example:\n' + records[0]['rejected'])

## 6. Load Qwen3-VL with Unsloth and attach the trainable SFT adapter

The policy starts from the SFT-408 adapter. Unsloth provides the optimized 4-bit Qwen3-VL loader and training patches; TRL still provides the multimodal DPO trainer/collator. `ref_model=None` makes TRL use the initial policy state as the DPO reference.

In [ ]:
from peft import PeftModel
from unsloth import FastVisionModel

if MODEL_BACKEND != 'UNSLOTH':
    raise ValueError(f'Unsupported MODEL_BACKEND={MODEL_BACKEND!r}')

base_model, processor = FastVisionModel.from_pretrained(
    model_name=UNSLOTH_BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)
processor.tokenizer.padding_side = 'left'
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_ID, is_trainable=True)
model.config.use_cache = False
FastVisionModel.for_training(model)
model.print_trainable_parameters()
print('Loaded trainable SFT-408 policy adapter with the Unsloth vision backend.')

## 7. Configure conservative multimodal DPO

The pilot uses one epoch, learning rate `2e-6`, sigmoid DPO, and `beta=0.1`. `max_length=None` is required so truncation cannot remove image tokens. Do not pass `UnslothVisionDataCollator`: it is an SFT collator, not the chosen/rejected DPO collator.

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_args = DPOConfig(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-6,
    warmup_steps=3,
    beta=0.1,
    loss_type='sigmoid',
    max_length=None,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit',
    max_grad_norm=1.0,
    bf16=BF16,
    fp16=not BF16,
    tf32=gpu.major >= 8,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    torch_empty_cache_steps=10,
    report_to='none',
    seed=SEED,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
)
print('Policy backend:', MODEL_BACKEND)
print('DPO collator:', type(trainer.data_collator).__name__)

## 8. Mandatory image-tensor preflight

Do not train if this cell fails. It directly verifies that TRL selected its vision preference collator and that the batch contains Qwen3-VL image tensors.

In [ ]:
import gc

collator_name = type(trainer.data_collator).__name__
if 'VisionPreference' not in collator_name:
    raise RuntimeError(
        f'STOPPED BEFORE TRAINING: expected TRL DataCollatorForVisionPreference, got {collator_name}.'
    )
batch = next(iter(trainer.get_train_dataloader()))
required_batch_keys = {'input_ids', 'attention_mask', 'completion_mask', 'pixel_values'}
missing_batch_keys = required_batch_keys - set(batch)
if missing_batch_keys:
    raise RuntimeError(
        f'STOPPED BEFORE TRAINING: multimodal batch missing {sorted(missing_batch_keys)}.'
    )
if batch['input_ids'].shape[0] != 2:
    raise RuntimeError('Expected chosen and rejected sequences in the DPO batch.')
print('VLM DPO preflight passed.')
print({key: tuple(value.shape) for key, value in batch.items() if hasattr(value, 'shape')})
del batch
gc.collect()
torch.cuda.empty_cache()

## 9. Train or resume

Checkpoints are stored in Drive. Re-running this cell resumes the latest trainer checkpoint when one exists.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

if not RUN_TRAINING:
    raise RuntimeError('RUN_TRAINING=False: preflight completed but training is intentionally disabled.')
last_checkpoint = (
    get_last_checkpoint(str(CHECKPOINT_DIR))
    if RESUME_FROM_LAST_CHECKPOINT and CHECKPOINT_DIR.exists()
    else None
)
print('Resume checkpoint:', last_checkpoint)
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
print(train_result.metrics)

## 10. Save the DPO-updated adapter and experiment manifest

In [ ]:
from datetime import datetime, timezone
from huggingface_hub import HfApi

trainer.save_model(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))
trainer.save_state()

api = HfApi()
def revision(repo_id):
    try:
        return api.model_info(repo_id).sha
    except Exception as error:
        warnings.warn(f'Could not resolve revision for {repo_id}: {error}')
        return None

manifest = {
    'experiment': 'FinChart Phase 2C multimodal DPO-386 provisional pilot',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'base_model': BASE_MODEL,
    'loaded_4bit_model': UNSLOTH_BASE_MODEL,
    'base_model_revision': revision(BASE_MODEL),
    'loaded_4bit_model_revision': revision(UNSLOTH_BASE_MODEL),
    'model_backend': MODEL_BACKEND,
    'source_sft_adapter': SFT_ADAPTER_ID,
    'source_sft_adapter_revision': revision(SFT_ADAPTER_ID),
    'pair_source': str(PAIR_JSONL),
    'pair_sha256': pair_sha256,
    'pairs_total': len(records),
    'pairs_train': len(train_dataset),
    'pairs_eval': len(eval_dataset),
    'manual_audit_status': dict(audit_statuses),
    'allow_provisional_pairs': ALLOW_PROVISIONAL_PAIRS,
    'allow_schema_mismatch': ALLOW_SCHEMA_MISMATCH,
    'same_field_signature': schema_matched,
    'algorithm': 'offline multimodal DPO',
    'loss_type': 'sigmoid',
    'beta': 0.1,
    'learning_rate': 2e-6,
    'epochs': 1,
    'effective_batch_size': 8,
    'train_metrics': train_result.metrics,
    'library_versions': {
        'unsloth': getattr(unsloth, '__version__', 'unknown'),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'trl': trl.__version__,
        'peft': peft.__version__,
        'datasets': datasets.__version__,
    },
    'phase1_validation_used_for_training': False,
    'reportable_model': pending_count == 0 and schema_matched == len(records),
    'next_step': 'Run the exact frozen Phase 1 evaluator on ChartQA val[0:500] and compare with SFT-408.',
}
manifest_path = RUN_ROOT / 'dpo_386_training_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print('Saved adapter:', ADAPTER_DIR)
print('Saved manifest:', manifest_path)

## 11. Optional Hugging Face upload

Keep `PUSH_TO_HUB=False` for a provisional diagnostic unless the repository name clearly marks it as non-reportable. Store `HF_TOKEN` in Colab Secrets; never paste it into the notebook.

In [ ]:
if PUSH_TO_HUB:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get('HF_TOKEN')
    if not hf_token:
        raise RuntimeError('Add HF_TOKEN to Colab Secrets before enabling PUSH_TO_HUB.')
    login(token=hf_token, add_to_git_credential=False)
    trainer.model.push_to_hub(HUB_MODEL_ID, token=hf_token, safe_serialization=True)
    processor.push_to_hub(HUB_MODEL_ID, token=hf_token)
    api.upload_file(
        path_or_fileobj=str(manifest_path),
        path_in_repo='dpo_386_training_manifest.json',
        repo_id=HUB_MODEL_ID,
        repo_type='model',
        token=hf_token,
    )
    print('Uploaded:', HUB_MODEL_ID)
else:
    print('Hub upload disabled; adapter remains in Google Drive.')

## 12. Required evaluation handoff

A completed training loss is not evidence of model improvement. Evaluate the saved adapter with the exact frozen Phase 1 protocol and report paired transitions against SFT-408:

```text
SFT-408: 345/500 (69.0%)
vs.
SFT-408 + DPO-386: pending frozen val[0:500] evaluation
```

Track overall accuracy, both-correct, DPO fixes, DPO regressions, both-wrong, and numerical/counting/logical/visual failure changes. A provisional-pair run must remain labeled diagnostic even if its score improves.